# Feature Engineering – Lags & Aggregations

## Overview
This notebook builds the base feature engineering layer for the modeling pipeline.

It performs the following steps:

1. Creates the base weekly sales table.
2. Generates dynamic lag features using a stored procedure.
3. Computes rolling aggregate features.
4. Prepares the dataset for downstream model training.

All transformation logic is encapsulated in stored procedures to allow
external orchestration (Task or external orchestrator) without modifying the core logic.


## Configuration Parameters

In [ ]:
-- -----------------------------------------------------------------------------
-- ENVIRONMENT CONFIGURATION
-- -----------------------------------------------------------------------------

USE ROLE AA_DEV_BMX_PS_DEVELOPER;
USE WAREHOUSE WH_AA_DEV_DS_SQL;
USE DATABASE BD_AA_DEV;
USE SCHEMA SC_STORAGE_BMX_PS;

-- Parameter: Number of lag weeks to generate
SET N_WEEKS = 4;


## 1. Create Base Weekly Sales Table

This step creates the base table used for feature generation.
It extracts the minimal required columns from the structured training dataset.


In [ ]:
-- -----------------------------------------------------------------------------
-- STEP 1: Create Base Table
-- This table serves as the source for lag and rolling feature generation.
-- -----------------------------------------------------------------------------

CREATE OR REPLACE TABLE WEEKLY_SALES_BASE AS
SELECT
    CUSTOMER_ID,
    WEEK,
    BRAND_PRES_RET,
    UNI_BOX_WEEK
FROM BD_AA_DEV.SC_STORAGE_BMX_PS.TRAIN_DATASET_STRUCTURED;


## 2. Create Stored Procedure – Dynamic Lags

This stored procedure dynamically generates lag features based on the number
of weeks defined in the input parameter (N_WEEKS).

Each lag represents historical sales values relative to the current week.


In [ ]:
-- -----------------------------------------------------------------------------
-- Stored Procedure: CREATE_SALES_LAGS_DYNAMIC
-- Purpose:
--   Dynamically generate N weekly lag features.
-- 
-- Input:
--   N_WEEKS → Number of historical weeks to calculate.
--
-- Output:
--   Creates or replaces table SALES_LAGS_WEEKLY.
--
-- Design Note:
--   Encapsulated to allow external orchestration.
-- -----------------------------------------------------------------------------

CREATE OR REPLACE PROCEDURE CREATE_SALES_LAGS_DYNAMIC(N_WEEKS INTEGER)
RETURNS STRING
LANGUAGE SQL
 AS $$
DECLARE
    sql_query STRING;
    lag_columns STRING DEFAULT '';
    i INTEGER;
BEGIN
    i := 1;
    WHILE (i <= N_WEEKS) DO
        lag_columns := lag_columns || 
        '    
        CASE 
            WHEN LAG(week, ' || i || ') OVER (PARTITION BY customer_id, brand_pres_ret ORDER BY week) = week - ' || i || '
            THEN COALESCE(LAG(uni_box_week, ' || i || ', 0) OVER (PARTITION BY customer_id, brand_pres_ret ORDER BY week), 0)
            ELSE 0
        END AS w_m' || i || '_total';
        
        IF (i < N_WEEKS) THEN
            lag_columns := lag_columns || ',\n';
        END IF;
        
        i := i + 1;
    END WHILE;
    
    sql_query := 
    'CREATE OR REPLACE TABLE SALES_LAGS_WEEKLY AS
    SELECT 
        customer_id,
        week,
        brand_pres_ret,
        uni_box_week,
        
        -- Lags: sales from previous weeks
    ' || lag_columns || '
        
    FROM weekly_sales_base
    ORDER BY customer_id, brand_pres_ret, week';
    
    EXECUTE IMMEDIATE sql_query;
    
    RETURN 'Table successfully created with ' || N_WEEKS || ' lag weeks';
END;
$$;

CALL CREATE_SALES_LAGS_DYNAMIC(4);


## 3. Create Stored Procedure – Rolling Aggregates

This procedure calculates rolling window statistics (e.g., averages)
to capture short-term trends in sales behavior.


In [ ]:
-- -----------------------------------------------------------------------------
-- Stored Procedure: CALCULATE_AGGREGATES
--
-- Purpose:
--   Compute rolling statistical features (e.g., previous 2-week average).
--
-- Output:
--   Creates or replaces table SALES_AGGREGATES.
--
-- Notes:
--   Rolling features are calculated only when a complete window exists
--   to avoid data leakage.
-- -----------------------------------------------------------------------------

CREATE OR REPLACE PROCEDURE CALCULATE_AGGREGATES()
RETURNS STRING
LANGUAGE SQL
AS
$$
BEGIN

    -- Create or replace table with rolling aggregates
    CREATE OR REPLACE TABLE SALES_AGGREGATES AS
    SELECT
        CUSTOMER_ID,
        WEEK,
        BRAND_PRES_RET,
        UNI_BOX_WEEK,

        -- AVG_PREV2: Average of previous 2 weeks (only if complete window exists)
        CASE
            WHEN COUNT(UNI_BOX_WEEK) OVER (
                PARTITION BY CUSTOMER_ID, BRAND_PRES_RET
                ORDER BY WEEK
                ROWS BETWEEN 2 PRECEDING AND 1 PRECEDING
            ) = 2
            THEN AVG(UNI_BOX_WEEK) OVER (
                PARTITION BY CUSTOMER_ID, BRAND_PRES_RET
                ORDER BY WEEK
                ROWS BETWEEN 2 PRECEDING AND 1 PRECEDING
            )
            ELSE 0
        END AS AVG_PREV2,

        -- MAX_PREV3: Maximum of previous 3 weeks
        CASE
            WHEN COUNT(UNI_BOX_WEEK) OVER (
                PARTITION BY CUSTOMER_ID, BRAND_PRES_RET
                ORDER BY WEEK
                ROWS BETWEEN 3 PRECEDING AND 1 PRECEDING
            ) = 3
            THEN MAX(UNI_BOX_WEEK) OVER (
                PARTITION BY CUSTOMER_ID, BRAND_PRES_RET
                ORDER BY WEEK
                ROWS BETWEEN 3 PRECEDING AND 1 PRECEDING
            )
            ELSE 0
        END AS MAX_PREV3,

        -- SUM_PAST_4_WEEKS: Sum of previous 4 weeks
        CASE
            WHEN COUNT(UNI_BOX_WEEK) OVER (
                PARTITION BY CUSTOMER_ID, BRAND_PRES_RET
                ORDER BY WEEK
                ROWS BETWEEN 4 PRECEDING AND 1 PRECEDING
            ) = 4
            THEN SUM(UNI_BOX_WEEK) OVER (
                PARTITION BY CUSTOMER_ID, BRAND_PRES_RET
                ORDER BY WEEK
                ROWS BETWEEN 4 PRECEDING AND 1 PRECEDING
            )
            ELSE 0
        END AS SUM_PAST_4_WEEKS,

        -- AVG_PAST_12_WEEKS: Average of previous 12 weeks
        CASE
            WHEN COUNT(UNI_BOX_WEEK) OVER (
                PARTITION BY CUSTOMER_ID, BRAND_PRES_RET
                ORDER BY WEEK
                ROWS BETWEEN 12 PRECEDING AND 1 PRECEDING
            ) = 12
            THEN AVG(UNI_BOX_WEEK) OVER (
                PARTITION BY CUSTOMER_ID, BRAND_PRES_RET
                ORDER BY WEEK
                ROWS BETWEEN 12 PRECEDING AND 1 PRECEDING
            )
            ELSE 0
        END AS AVG_PAST_12_WEEKS

    FROM SALES_LAGS_WEEKLY
    ORDER BY CUSTOMER_ID, BRAND_PRES_RET, WEEK;

    RETURN 'SUCCESS';

EXCEPTION
    WHEN OTHER THEN
        RETURN 'PROCESSING ERROR: ' || SQLERRM;

END;
$$;

CALL CALCULATE_AGGREGATES();


In [ ]:
-- -----------------------------------------------------------------------------

-- ## 4. Final Training Dataset

-- This step combines lag features and aggregate features into a final dataset
-- used for model training.
-- -----------------------------------------------------------------------------

CREATE OR REPLACE PROCEDURE EXECUTE_LAGS_AGGREGATES(N_WEEKS INTEGER)
RETURNS STRING
LANGUAGE SQL
AS
$$
BEGIN

    CALL CREATE_SALES_LAGS_DYNAMIC(N_WEEKS);
    CALL CALCULATE_AGGREGATES();

    CREATE OR REPLACE TABLE TRAIN_DATA AS
    SELECT DISTINCT
        SA.CUSTOMER_ID,
        SA.BRAND_PRES_RET,
        SA.WEEK,
        SA.UNI_BOX_WEEK,
        SLW.W_M1_TOTAL,
        SLW.W_M2_TOTAL,
        SLW.W_M3_TOTAL,
        SLW.W_M4_TOTAL,
        SA.AVG_PREV2,
        SA.MAX_PREV3,
        SA.SUM_PAST_4_WEEKS,
        SA.AVG_PAST_12_WEEKS
    FROM SALES_AGGREGATES SA
    INNER JOIN SALES_LAGS_WEEKLY SLW
        ON SA.CUSTOMER_ID    = SLW.CUSTOMER_ID
       AND SA.BRAND_PRES_RET = SLW.BRAND_PRES_RET
       AND SA.WEEK           = SLW.WEEK;

    RETURN 'PROCESS COMPLETED SUCCESSFULLY';

EXCEPTION
    WHEN OTHER THEN
        RETURN 'PROCESS ERROR: ' || SQLERRM;

END;
$$;

CALL EXECUTE_LAGS_AGGREGATES(4)


## Architecture Note

All feature engineering logic is encapsulated in stored procedures.

This allows:
- Execution via external orchestrator (e.g., Dagster).